In [ ]:
from datetime import datetime
from pathlib import Path
import logging
from huggingface_hub import login, whoami
from transformers.utils.logging import disable_progress_bar
import os

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner, APIType
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.utils.logging_setup import install_colored_logger, setup_log_file_handler
from gsm_benchmarker.utils.seeds import set_seed


disable_progress_bar()
set_seed(42)


hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']
login(hf_api_token)

whoami()['name']

openai package not installed; OpenAI models will not be available
anthropic package not installed; Anthropic models will not be available
google-genai package not installed; Gemini family models will not be available


In [ ]:
OUTPUT_ROOT_PATH = Path(f"../../../data/gsm-symbolic").resolve()
RESULTS_ROOT_PATH = OUTPUT_ROOT_PATH / "outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(OUTPUT_ROOT_PATH)
print(os.environ['HF_HOME'])

/home/guests2/dkd/data/gsm-symbolic/outputs/20251029_110755
/media/generalstorage4/dkdstorage


In [ ]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root', 'bitsandbytes', 'transformers_modules'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

install_colored_logger(level=logging.INFO)
setup_log_file_handler(OUTPUT_ROOT_PATH / "logs")

NameError: name 'setup_log_file_handler' is not defined

In [ ]:
models_config = ModelsConfig()
all_models = models_config.all_models
print(models_config.all_models)  # note: openai - insufficient quota for benchmarking

[('google/gemma-2b', None), ('google/gemma-2b-it', None), ('google/gemma-7b', None), ('google/gemma-7b-it', None), ('google/gemma-2-2b', None), ('google/gemma-2-2b-it', None), ('google/gemma-2-9b', None), ('google/gemma-2-9b-it', None), ('google/gemma-2-27b-it', None), ('microsoft/phi-2', None), ('microsoft/Phi-3-mini-128k-instruct', None), ('microsoft/Phi-3-small-128k-instruct', None), ('microsoft/Phi-3-medium-128k-instruct', None), ('microsoft/Phi-3.5-mini-instruct', None), ('mistralai/Mistral-7B-v0.1', None), ('mistralai/Mistral-7B-Instruct-v0.1', None), ('mistralai/Mistral-7B-v0.3', None), ('mistralai/Mistral-7B-Instruct-v0.3', None), ('mistralai/Mathstral-7B-v0.1', None), ('meta-llama/Meta-Llama-3-8B', None), ('meta-llama/Meta-Llama-3-8B-Instruct', None), ('gpt-4o-mini', <APIType.openai: 1>), ('gpt-4o', <APIType.openai: 1>), ('o1-mini', <APIType.openai: 1>), ('o1-preview', <APIType.openai: 1>)]


In [ ]:
open_models = models_config.open_models
print(open_models)

['google/gemma-2b', 'google/gemma-2b-it', 'google/gemma-7b', 'google/gemma-7b-it', 'google/gemma-2-2b', 'google/gemma-2-2b-it', 'google/gemma-2-9b', 'google/gemma-2-9b-it', 'google/gemma-2-27b-it', 'microsoft/phi-2', 'microsoft/Phi-3-mini-128k-instruct', 'microsoft/Phi-3-small-128k-instruct', 'microsoft/Phi-3-medium-128k-instruct', 'microsoft/Phi-3.5-mini-instruct', 'mistralai/Mistral-7B-v0.1', 'mistralai/Mistral-7B-Instruct-v0.1', 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3', 'mistralai/Mathstral-7B-v0.1', 'meta-llama/Meta-Llama-3-8B', 'meta-llama/Meta-Llama-3-8B-Instruct']


In [ ]:
# these are not in the paper, but interestingly give very good results (at least on a small subset of the benchmark data, main variant)
# bigger quota than for openAI models, but likely still too small for full benchmark
other_closed_models = [
    ("gemini-2.0-flash", APIType.google_genai),
    ("gemini-2.5-flash", APIType.google_genai),
]

In [ ]:
variants = GSMSymbolicDataset.Variant

# br = BenchmarkRunner(
#     models=open_models[9:],
#     dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
#     storage_path==RESULTS_ROOT_PATH,
#     config=BenchmarkConfig(trust_remote_code=True, gpu0_max_memory="14GB", cpu_max_memory="10GB")
# )

In [ ]:
# br.run(n_sets=2, n_per_set=2)

In [ ]:
failed_models = ['microsoft/Phi-3-small-128k-instruct']  #, 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3']
br2 = BenchmarkRunner(
    models=failed_models,
    dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
    storage_path=RESULTS_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code=True, gpu0_max_memory="14GB", cpu_max_memory="10GB")
)
br2.run(n_sets=2, n_per_set=2)

gsm_benchmarker.utils.path_ops [INFO] 2025-10-29 11:07:55: Creating directories at /home/guests2/dkd/data/gsm-symbolic/outputs/20251029_110755
gsm_benchmarker.benchmark [INFO] 2025-10-29 11:07:55: ========== Evaluating model 1/1: microsoft/Phi-3-small-128k-instruct ==========
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-29 11:07:55: Setting up model microsoft/Phi-3-small-128k-instruct
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-small-128k-instruct:
- tokenization_phi3_small.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-29 11:07:56: CUDA available
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-small-128k-instruct:
- configuration_phi3_small.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloa

set:   0%|          | 0/1 [00:00<?, ?it/s]

gsm_benchmarker.utils.path_ops [INFO] 2025-10-29 11:15:11: Creating directories at /home/guests2/dkd/data/gsm-symbolic/outputs/20251029_110755/intermediate
gsm_benchmarker.model_evaluator [INFO] 2025-10-29 11:15:11: Intermediate results will be stored at: /home/guests2/dkd/data/gsm-symbolic/outputs/20251029_110755/intermediate/microsoft_Phi-3-small-128k-instruct_20251029_111511


Dataset:   0%|          | 0/1 [00:00<?, ?it/s]

Example:   0%|          | 0/2 [00:00<?, ?it/s]

...

Example:   0%|          | 0/2 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [ERROR] 2025-10-29 11:15:18: Exception occurred when evaluating dataset 1/2: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
. Results for this dataset will be skipped.
gsm_benchmarker.model_evaluator [ERROR] 2025-10-29 11:15:18: Full stack:
Traceback (most recent call last):
  File "/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/model_evaluator.py", line 195, in evaluate_multiple_datasets
    result = self.evaluate_dataset(dataset)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/model_evaluator.py", line 1

Example:   0%|          | 0/2 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [ERROR] 2025-10-29 11:15:18: Exception occurred when evaluating dataset 2/2: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
. Results for this dataset will be skipped.
gsm_benchmarker.model_evaluator [ERROR] 2025-10-29 11:15:18: Full stack:
Traceback (most recent call last):
  File "/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/model_evaluator.py", line 195, in evaluate_multiple_datasets
    result = self.evaluate_dataset(dataset)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/model_evaluator.py", line 1

{<Variant.GSM8K: 4>: {'microsoft/Phi-3-small-128k-instruct': None},
 <Variant.main: 1>: {'microsoft/Phi-3-small-128k-instruct': None},
 <Variant.p1: 2>: {'microsoft/Phi-3-small-128k-instruct': None},
 <Variant.p2: 3>: {'microsoft/Phi-3-small-128k-instruct': None}}

In [ ]:
print(br2.summarise_failures())

Recorded 4 failures:

On model microsoft/Phi-3-small-128k-instruct, variant GSM8K - failed to evaluate: Encountered 1 error(s() when evaluating microsoft/Phi-3-small-128k-instruct on variant Variant.GSM8K; first one was: FlashAttention only supports Ampere GPUs or newer.

On model microsoft/Phi-3-small-128k-instruct, variant main - failed to evaluate: Encountered 2 error(s() when evaluating microsoft/Phi-3-small-128k-instruct on variant Variant.main; first one was: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


On model microsoft/Phi-3-small-128k-instruct, variant p1 - failed to evaluate: Encountered 2 error(s() when evaluating 

In [ ]:
br.summarise_results()

NameError: name 'br' is not defined

In [ ]:
el = br.results[GSMSymbolicDataset.Variant.main][open_models[-1]].loc[0, 0]
el

In [ ]:
print(el.question)
print()
print(el.response)